# 05 - Treinamento ResNet50 - Comparação Justa 


Este notebook usa a mesma configuração experimental aplicada aos três modelos:

- divisão 60% treino / 20% validação / 20% teste;
- imagens em 224 x 224;
- batch size 16;
- data augmentation geométrico moderado aplicado somente durante o treinamento;
- preprocessamento específico da arquitetura;
- pesos por classe calculados somente no conjunto de treinamento;
- treinamento da cabeça classificadora;
- fine-tuning das últimas 50 camadas;
- callbacks monitorando `val_loss`;
- avaliação final somente no conjunto de teste.

Versão corrigida: inclui definição de `DROPOUT_RATE`, reconstrução dos caminhos das imagens a partir da pasta atual do projeto e aplicação explícita do `preprocess_input` após o aumento de dados.


In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

print("TensorFlow:", tf.__version__)
print("GPUs disponíveis:", tf.config.list_physical_devices("GPU"))


In [ ]:
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# Localiza a raiz do projeto de forma mais robusta.
# Funciona tanto quando o notebook é executado dentro de notebooks/ quanto pela raiz do projeto.
def find_project_dir():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate.resolve()
    return Path("..").resolve()

PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "data"
RAW_IMAGES_DIR = DATA_DIR / "raw" / "train_images"
SPLITS_DIR = DATA_DIR / "splits"

RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
LOGS_DIR = RESULTS_DIR / "logs"
MODELS_DIR = PROJECT_DIR / "models"

for directory in [FIGURES_DIR, METRICS_DIR, LOGS_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "ResNet50"
MODEL_KEY = "resnet50"
EXPERIMENT_NAME = "v3_comparacao_justa_ft50_lr1e5"
MODEL_OUTPUT_KEY = f"{MODEL_KEY}_{EXPERIMENT_NAME}"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
NUM_CLASSES = 5
AUTOTUNE = tf.data.AUTOTUNE

EPOCHS_HEAD = 10
HEAD_LEARNING_RATE = 1e-3

RUN_FINE_TUNING = True
EPOCHS_FINE_TUNING = 20
FINE_TUNING_LEARNING_RATE = 1e-5
FINE_TUNE_LAST_N_LAYERS = 50

DROPOUT_RATE = 0.3

CLASS_WEIGHT_MODE = "balanced"  # opções: "balanced", "sqrt", "none"

class_names = {
    0: "Sem retinopatia",
    1: "Retinopatia leve",
    2: "Retinopatia moderada",
    3: "Retinopatia severa",
    4: "Retinopatia proliferativa"
}

print("Raiz do projeto:", PROJECT_DIR)
print("Pasta das imagens:", RAW_IMAGES_DIR)
print("Modelo:", MODEL_NAME)
print("Experimento:", EXPERIMENT_NAME)
print("Tamanho da imagem:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Dropout:", DROPOUT_RATE)
print("Fine-tuning últimas camadas:", FINE_TUNE_LAST_N_LAYERS)
print("Modo de class_weight:", CLASS_WEIGHT_MODE)


## Leitura dos arquivos de divisão


In [ ]:
train_df = pd.read_csv(SPLITS_DIR / "train_split.csv")
val_df = pd.read_csv(SPLITS_DIR / "val_split.csv")
test_df = pd.read_csv(SPLITS_DIR / "test_split.csv")

# Reconstrói os caminhos das imagens a partir da pasta atual do projeto.
# Isso evita erro quando os CSVs foram gerados em outra máquina ou outro diretório.
def corrigir_caminhos(df):
    df = df.copy()
    df["image_path"] = df["id_code"].astype(str).apply(
        lambda image_id: str((RAW_IMAGES_DIR / f"{image_id}.png").resolve())
    )
    return df

train_df = corrigir_caminhos(train_df)
val_df = corrigir_caminhos(val_df)
test_df = corrigir_caminhos(test_df)

print("Treino:", train_df.shape)
print("Validação:", val_df.shape)
print("Teste:", test_df.shape)

print("\nExemplo de caminho de imagem:")
print(train_df["image_path"].iloc[0])

display(train_df.head())


In [ ]:
def verificar_imagens(df, nome):
    missing = df[~df["image_path"].apply(lambda path: Path(path).exists())]

    print(f"{nome}:")
    print("Total:", len(df))
    print("Imagens ausentes:", len(missing))

    if len(missing) > 0:
        display(missing.head())
        raise FileNotFoundError(f"Existem imagens ausentes em {nome}. Verifique data/raw/train_images.")

verificar_imagens(train_df, "Treino")
verificar_imagens(val_df, "Validação")
verificar_imagens(test_df, "Teste")


In [ ]:
print("Distribuição no treino:")
print(train_df["diagnosis"].value_counts().sort_index())

print("\nDistribuição na validação:")
print(val_df["diagnosis"].value_counts().sort_index())

print("\nDistribuição no teste:")
print(test_df["diagnosis"].value_counts().sort_index())

print("\nPercentual no treino:")
print((train_df["diagnosis"].value_counts(normalize=True).sort_index() * 100).round(2))

print("\nPercentual na validação:")
print((val_df["diagnosis"].value_counts(normalize=True).sort_index() * 100).round(2))

print("\nPercentual no teste:")
print((test_df["diagnosis"].value_counts(normalize=True).sort_index() * 100).round(2))


## Pesos por classe


In [ ]:
classes = np.unique(train_df["diagnosis"].values)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["diagnosis"].values
)

class_weights_balanced = {
    int(class_id): float(weight)
    for class_id, weight in zip(classes, class_weights_values)
}

class_weights_sqrt = {
    class_id: float(np.sqrt(weight))
    for class_id, weight in class_weights_balanced.items()
}

if CLASS_WEIGHT_MODE == "balanced":
    class_weights = class_weights_balanced
elif CLASS_WEIGHT_MODE == "sqrt":
    class_weights = class_weights_sqrt
elif CLASS_WEIGHT_MODE == "none":
    class_weights = None
else:
    raise ValueError("CLASS_WEIGHT_MODE deve ser 'balanced', 'sqrt' ou 'none'.")

print("Class weights balanceados:")
print(class_weights_balanced)

print("\nClass weights suavizados:")
print(class_weights_sqrt)

print("\nClass weights usados neste experimento:")
print(class_weights)


## Pipeline TensorFlow


In [ ]:
def load_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)

    # Importante: não dividir por 255 aqui.
    # O preprocess_input será aplicado dentro do modelo, após o data augmentation.
    label = tf.one_hot(label, NUM_CLASSES)

    return image, label


def make_dataset(df, shuffle=False):
    # Converte os caminhos para string e normaliza separadores no Windows.
    image_paths = (
        df["image_path"]
        .astype(str)
        .str.replace("\\", "/", regex=False)
        .values
    )

    labels = df["diagnosis"].astype(int).values

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(df),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

    return ds


train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)
test_ds = make_dataset(test_df, shuffle=False)

for images, labels in train_ds.take(1):
    print("Treino - imagens:", images.shape)
    print("Treino - rótulos:", labels.shape)
    print("Treino - dtype imagens:", images.dtype)
    print("Treino - dtype rótulos:", labels.dtype)

for images, labels in val_ds.take(1):
    print("Validação - imagens:", images.shape)
    print("Validação - rótulos:", labels.shape)

for images, labels in test_ds.take(1):
    print("Teste - imagens:", images.shape)
    print("Teste - rótulos:", labels.shape)


## Data augmentation


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.03, fill_mode="constant", fill_value=0.0),
    tf.keras.layers.RandomZoom(0.05, fill_mode="constant", fill_value=0.0),
    tf.keras.layers.RandomTranslation(0.03, 0.03, fill_mode="constant", fill_value=0.0),
    tf.keras.layers.RandomFlip("horizontal"),
], name="data_augmentation")


## Criação do modelo


In [ ]:
base_model = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

base_model.trainable = False

inputs = tf.keras.Input(
    shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    name="input_image"
)

# O aumento de dados fica dentro do modelo.
# Ele é aplicado automaticamente apenas durante o treinamento.
x = data_augmentation(inputs)

# O preprocess_input específico da ResNet50 é aplicado depois do augmentation.
# A imagem chega ao modelo em float32, na escala 0-255, e é adaptada aqui.
x = tf.keras.layers.Lambda(
    lambda image: preprocess_input(image),
    name=f"{MODEL_KEY}_preprocess_input"
)(x)

x = base_model(x, training=False)

x = tf.keras.layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = tf.keras.layers.Dropout(
    DROPOUT_RATE,
    name="dropout"
)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="classification_output"
)(x)

model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name=f"{MODEL_NAME}_APTOS_{EXPERIMENT_NAME}"
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=HEAD_LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
# Teste rápido do fluxo completo antes do model.fit.
# Se esta célula rodar, os dados e o modelo estão compatíveis.
for images, labels in train_ds.take(1):
    print("Imagens:", images.shape)
    print("Rótulos:", labels.shape)
    print("Tipo das imagens:", images.dtype)
    print("Tipo dos rótulos:", labels.dtype)

    preds = model(images, training=True)
    print("Predições:", preds.shape)


## Callbacks


In [ ]:
MODEL_OUTPUT_DIR = MODELS_DIR / MODEL_OUTPUT_KEY
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = MODEL_OUTPUT_DIR / f"{MODEL_OUTPUT_KEY}_best.keras"
final_model_path = MODEL_OUTPUT_DIR / f"{MODEL_OUTPUT_KEY}_final.keras"

head_csv_log_path = LOGS_DIR / f"{MODEL_OUTPUT_KEY}_head_training_log.csv"
fine_csv_log_path = LOGS_DIR / f"{MODEL_OUTPUT_KEY}_fine_tuning_log.csv"

def build_callbacks(csv_log_path):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor="val_loss",
            save_best_only=True,
            mode="min",
            verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.2,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
        tf.keras.callbacks.CSVLogger(
            filename=str(csv_log_path),
            append=False
        )
    ]

callbacks_head = build_callbacks(head_csv_log_path)
callbacks_fine = build_callbacks(fine_csv_log_path)

print("Diretório do modelo:", MODEL_OUTPUT_DIR)
print("Checkpoint:", checkpoint_path)
print("Log cabeça:", head_csv_log_path)
print("Log fine-tuning:", fine_csv_log_path)


## Etapa 1: treinamento da cabeça classificadora


In [ ]:
start_time = time.time()

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    class_weight=class_weights,
    callbacks=callbacks_head
)

head_training_time = time.time() - start_time

print(f"Tempo de treinamento da cabeça: {head_training_time:.2f} segundos")


## Etapa 2: fine-tuning


In [ ]:
history_fine = None
fine_tuning_time = 0.0

if RUN_FINE_TUNING:
    base_model.trainable = True

    for layer in base_model.layers[:-FINE_TUNE_LAST_N_LAYERS]:
        layer.trainable = False

    # Mantém BatchNormalization congeladas para maior estabilidade.
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNING_LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    print("Fine-tuning ativado.")
    print("Últimas camadas liberadas:", FINE_TUNE_LAST_N_LAYERS)
    print("Learning rate:", FINE_TUNING_LEARNING_RATE)

    start_time = time.time()

    history_fine = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FINE_TUNING,
        class_weight=class_weights,
        callbacks=callbacks_fine
    )

    fine_tuning_time = time.time() - start_time

    print(f"Tempo de fine-tuning: {fine_tuning_time:.2f} segundos")
else:
    print("Fine-tuning desativado.")


## Histórico de treinamento


In [ ]:
history_parts = []

history_head_df = pd.DataFrame(history_head.history)
history_head_df["phase"] = "head"
history_head_df["phase_epoch"] = np.arange(1, len(history_head_df) + 1)
history_parts.append(history_head_df)

if history_fine is not None:
    history_fine_df = pd.DataFrame(history_fine.history)
    history_fine_df["phase"] = "fine_tuning"
    history_fine_df["phase_epoch"] = np.arange(1, len(history_fine_df) + 1)
    history_parts.append(history_fine_df)

history_df = pd.concat(history_parts, ignore_index=True)
history_df["global_epoch"] = np.arange(1, len(history_df) + 1)

history_path = METRICS_DIR / f"{MODEL_OUTPUT_KEY}_history.csv"
history_df.to_csv(history_path, index=False, encoding="utf-8-sig")

print("Histórico salvo em:", history_path)
display(history_df.tail())


## Curvas de treinamento


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_df["global_epoch"], history_df["loss"], label="Loss treino")

if "val_loss" in history_df.columns:
    plt.plot(history_df["global_epoch"], history_df["val_loss"], label="Loss validação")

plt.title(f"Loss durante o treinamento - {MODEL_NAME}")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

loss_fig_path = FIGURES_DIR / f"{MODEL_OUTPUT_KEY}_loss.png"
plt.savefig(loss_fig_path, dpi=300)
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(history_df["global_epoch"], history_df["accuracy"], label="Acurácia treino")

if "val_accuracy" in history_df.columns:
    plt.plot(history_df["global_epoch"], history_df["val_accuracy"], label="Acurácia validação")

plt.title(f"Acurácia durante o treinamento - {MODEL_NAME}")
plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.legend()
plt.grid(True)
plt.tight_layout()

acc_fig_path = FIGURES_DIR / f"{MODEL_OUTPUT_KEY}_accuracy.png"
plt.savefig(acc_fig_path, dpi=300)
plt.show()

print("Figura de loss salva em:", loss_fig_path)
print("Figura de acurácia salva em:", acc_fig_path)


## Avaliação no conjunto de teste


In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Loss no teste:", test_loss)
print("Acurácia no teste:", test_accuracy)


## Predições no conjunto de teste


In [ ]:
y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)

    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Total de amostras avaliadas:", len(y_true))


## Relatório de classificação


In [ ]:
target_names = [class_names[i] for i in range(NUM_CLASSES)]

report = classification_report(
    y_true,
    y_pred,
    target_names=target_names,
    digits=4
)

report_path = METRICS_DIR / f"{MODEL_OUTPUT_KEY}_classification_report.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print(report)
print("Relatório salvo em:", report_path)


## Matriz de confusão


In [ ]:
cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=target_names,
    columns=target_names
)

cm_path = METRICS_DIR / f"{MODEL_OUTPUT_KEY}_confusion_matrix.csv"
cm_df.to_csv(cm_path, encoding="utf-8-sig")

display(cm_df)
print("Matriz de confusão salva em:", cm_path)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_names
)

fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, values_format="d", xticks_rotation=45)
plt.title(f"Matriz de Confusão - {MODEL_NAME}")
plt.tight_layout()

cm_fig_path = FIGURES_DIR / f"{MODEL_OUTPUT_KEY}_confusion_matrix.png"
plt.savefig(cm_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Figura da matriz salva em:", cm_fig_path)


## Salvamento do modelo e resumo do experimento


In [ ]:
model.save(final_model_path)

summary_metrics = {
    "model": MODEL_NAME,
    "experiment_name": EXPERIMENT_NAME,
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "epochs_head_configured": int(EPOCHS_HEAD),
    "epochs_fine_tuning_configured": int(EPOCHS_FINE_TUNING) if RUN_FINE_TUNING else 0,
    "epochs_completed_total": int(len(history_df)),
    "batch_size": int(BATCH_SIZE),
    "image_size": list(IMG_SIZE),
    "dropout_rate": float(DROPOUT_RATE),
    "class_weight_mode": CLASS_WEIGHT_MODE,
    "class_weights": class_weights,
    "split_strategy": "60% treino / 20% validação / 20% teste",
    "preprocess_input": "tensorflow.keras.applications.efficientnet.preprocess_input",
    "data_augmentation": {
        "RandomRotation": 0.03,
        "RandomZoom": 0.05,
        "RandomTranslation": [0.03, 0.03],
        "RandomFlip": "horizontal"
    },
    "run_fine_tuning": bool(RUN_FINE_TUNING),
    "fine_tune_last_n_layers": int(FINE_TUNE_LAST_N_LAYERS) if RUN_FINE_TUNING else 0,
    "head_learning_rate": float(HEAD_LEARNING_RATE),
    "fine_tuning_learning_rate": float(FINE_TUNING_LEARNING_RATE) if RUN_FINE_TUNING else None,
    "head_training_time_seconds": float(head_training_time),
    "fine_tuning_time_seconds": float(fine_tuning_time),
    "gpu_available": bool(tf.config.list_physical_devices("GPU")),
    "final_model_path": str(final_model_path),
    "best_checkpoint_path": str(checkpoint_path)
}

summary_path = METRICS_DIR / f"{MODEL_OUTPUT_KEY}_summary_metrics.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_metrics, f, indent=4, ensure_ascii=False)

print("Modelo final salvo em:", final_model_path)
print("Resumo salvo em:", summary_path)

summary_metrics
